# Radiomics Feature Extraction

Demonstrates PyRadiomics feature extraction from NIfTI volumes and integration with the CLARYON pipeline.

Requires: `pip install pyradiomics nibabel`

In [ ]:
# Check dependencies
HAS_DEPS = True
try:
    import nibabel as nib
    import radiomics
    print(f"nibabel: {nib.__version__}")
    print(f"pyradiomics: {radiomics.__version__}")
except ImportError as e:
    HAS_DEPS = False
    print(f"Missing dependency: {e}")
    print("Install with: pip install pyradiomics nibabel")

In [ ]:
import tempfile
from pathlib import Path
import numpy as np

if HAS_DEPS:
    # Create synthetic NIfTI volumes with masks
    tmp = tempfile.mkdtemp()
    img_dir = Path(tmp) / "images"
    img_dir.mkdir()

    pairs = []
    for i in range(6):
        vol = (np.random.rand(32, 32, 32) * 100).astype(np.float32)
        mask = np.zeros((32, 32, 32), dtype=np.int16)
        mask[8:24, 8:24, 8:24] = 1  # Central ROI

        img_path = img_dir / f"patient_{i:03d}_image.nii.gz"
        mask_path = img_dir / f"patient_{i:03d}_mask.nii.gz"
        nib.save(nib.Nifti1Image(vol, np.eye(4)), str(img_path))
        nib.save(nib.Nifti1Image(mask, np.eye(4)), str(mask_path))
        pairs.append((str(img_path), str(mask_path)))

    print(f"Created {len(pairs)} image/mask pairs")

In [ ]:
if HAS_DEPS:
    from claryon.preprocessing.radiomics import extract_radiomics_batch

    radiomics_df = extract_radiomics_batch(pairs)
    print(f"Extracted {radiomics_df.shape[1]} radiomic features for {radiomics_df.shape[0]} subjects")
    display(radiomics_df.head())

In [ ]:
# Show YAML config for radiomics pipeline
config_example = """
data:
  imaging:
    path: data/nifti_dataset
    format: nifti
    image_pattern: "*image*"
    mask_pattern: "*mask*"
  radiomics:
    extract: true
    config: configs/pyradiomics_default.yaml

models:
  - name: xgboost
    type: tabular
"""
print("Radiomics pipeline config:")
print(config_example)